# Probe Training & Permutation Tests

Trains Probe 1 (Mode 1) and Probe 2 (Mode 2) on the ProScript v1a training set.
Layer selection uses held-out dev features. Permutation tests confirm signal validity.

**Required:** `train.jsonl`, `dev.jsonl`, `test.jsonl`

**Outputs:** `probe_manifest.json`, `probe1_layer{N}.pkl`, `probe2_layer{N}.pkl`,
layer sweep CSVs, permutation CSVs, figures.


In [ ]:
import subprocess, os
subprocess.run(['pip','install','-q','transformers','accelerate',
                'scikit-learn','tqdm','matplotlib'], check=True)
os.makedirs('/content/cache', exist_ok=True)
for f in ['train.jsonl','dev.jsonl','test.jsonl']:
    assert os.path.exists(f'/content/{f}'), f'Missing: {f}'
print('✓ All files present')


In [ ]:
import random, hashlib, json as _json
import numpy as np, pandas as pd
from pathlib import Path
CONTENT = Path('/content')

cfg = {
    'model_name':   'mistralai/Mistral-7B-Instruct-v0.1',
    'sweep_layers': list(range(12, 23)),
    'n_perms':      100,
    'n_cv_folds':   5,
    'chat_template':True,
    'seed':         42,
    'use_cache':    True,
}
SEED = cfg['seed']

# Issue 12: cache key tied to config that affects features
_cache_sig = hashlib.md5(
    _json.dumps({k: cfg[k] for k in
                 ['model_name','sweep_layers','chat_template','seed']},
                sort_keys=True).encode()
).hexdigest()[:10]
print(f'Config ready | sweep: {cfg["sweep_layers"]} | cache sig: {_cache_sig}')


In [ ]:
import json as _json
from collections import defaultdict, deque

def load_jsonl_full(path):
    plans, seen = {}, set()
    with open(path, encoding='utf-8-sig') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            d = _json.loads(line)
            goal = d.get('scenario') or d.get('goal') or d.get('id', '')
            # Issue 23: assert uniqueness
            if goal in seen:
                goal = f'{goal}__dup{i}'  # disambiguate
            seen.add(goal)
            plans[goal] = d
    return plans

def load_jsonl_goals(path):
    goals = set()
    with open(path, encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            d = _json.loads(line)
            g = d.get('scenario') or d.get('goal') or d.get('id','')
            if g: goals.add(g)
    return goals

def parse_plan(d):
    """Returns steps {id:text}, id_edges [(int,int)], adj {int:[int]}, real [int]."""
    steps = {int(k): v.strip() for k, v in d['steps'].items()}
    start = next((k for k,v in steps.items() if v.upper()=='START'), None)
    end   = next((k for k,v in steps.items() if v.upper()=='END'),   None)
    real  = [k for k in steps if k not in (start, end)]
    adj   = defaultdict(list)
    id_edges = []
    for a, b in d['edges']:
        a, b = int(a), int(b)
        if a in (start,end) or b in (start,end): continue
        if a in steps and b in steps:
            adj[a].append(b)
            id_edges.append((a, b))
    return steps, id_edges, adj, real

def _reach(start, adj):
    v, q = set(), [start]
    while q:
        n = q.pop()
        for nb in adj.get(n, []):
            if nb not in v: v.add(nb); q.append(nb)
    return v

def get_incomparable(real, adj):
    r = {n: _reach(n, adj) for n in real}
    return [(real[i], real[j])
            for i in range(len(real))
            for j in range(i+1, len(real))
            if real[j] not in r[real[i]] and real[i] not in r[real[j]]]

def compute_depths(real, adj):
    in_deg = {s:0 for s in real}
    for n in real:
        for nb in adj.get(n,[]): in_deg[nb]=in_deg.get(nb,0)+1
    depth = {s:0 for s in real}
    q = deque([s for s in real if in_deg.get(s,0)==0])
    while q:
        n = q.popleft()
        for nb in adj.get(n,[]):
            depth[nb] = max(depth[nb], depth[n]+1)
            in_deg[nb] -= 1
            if in_deg[nb]==0: q.append(nb)
    return depth

print('Utils ready')


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def wrap(raw):
    return f'[INST] {raw.strip()} [/INST]' if cfg['chat_template'] else raw

def tp1_ordering(goal, a, b):
    return wrap(
        f'You are judging a temporal dependency between two actions in a task.\n'
        f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
        f'Question: Must Action A happen before Action B? '
        f'Answer yes or no.\nAnswer:')

print(f'Loading {cfg["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
model = AutoModelForCausalLM.from_pretrained(
    cfg['model_name'], torch_dtype=torch.float16,
    device_map='auto', output_hidden_states=True)
model.eval()

# Issue 21: deduplicate token IDs, verify single-token encoding
def _single_token_ids(variants):
    ids = set()
    for text in variants:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
        else:
            print(f'  WARNING: "{text}" encodes to {len(enc)} tokens, skipped')
    return sorted(ids)

YES_IDS = _single_token_ids([' yes','yes','Yes',' Yes'])
NO_IDS  = _single_token_ids([' no', 'no', 'No', ' No'])
print(f'YES tokens: {YES_IDS} ({len(YES_IDS)} unique)')
print(f'NO  tokens: {NO_IDS}  ({len(NO_IDS)} unique)')


In [ ]:
import pickle

@torch.no_grad()
def get_all_layers_and_pyes(prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out    = model(**inputs, output_hidden_states=True)
    hiddens = [out.hidden_states[l][0,-1].float().cpu().numpy()
               for l in cfg['sweep_layers']]
    probs   = torch.softmax(out.logits[0,-1].float(), dim=-1).cpu()
    p_yes   = sum(probs[i].item() for i in YES_IDS)
    p_no    = sum(probs[i].item() for i in NO_IDS)
    denom   = p_yes + p_no if (p_yes + p_no) > 0 else 1.0
    return hiddens, p_yes / denom

# Issue 12: cache keyed by config signature
def _cpath(tag): return CONTENT / 'cache' / f'{tag}_{_cache_sig}.pkl'

def save_cache(tag, d):
    pickle.dump(d, open(_cpath(tag), 'wb'))
    print(f'  Cached: {_cpath(tag).name}')

def load_cache(tag):
    return pickle.load(open(_cpath(tag), 'rb'))

def has_cache(tag): return _cpath(tag).exists()

print('Feature helpers ready')


In [ ]:
from tqdm.auto import tqdm

train_plans = load_jsonl_full(CONTENT / 'train.jsonl')
dev_plans   = load_jsonl_full(CONTENT / 'dev.jsonl')
test_goals  = load_jsonl_goals(CONTENT / 'test.jsonl')
eval_goals  = set(dev_plans) | test_goals

def build_rows(plans, exclude=None):
    p1, p2 = [], []
    for goal, d in tqdm(plans.items(), desc='Building rows', leave=False):
        if exclude and goal in exclude: continue
        steps, id_edges, adj, real = parse_plan(d)
        if not real or not id_edges: continue
        depths = compute_depths(real, adj)
        incompat = get_incomparable(real, adj)
        # Use step TEXT for prompts, but keep goal for grouping
        for a_id, b_id in id_edges:
            a_t, b_t = steps[a_id], steps[b_id]
            p1.append({'goal':goal,'a':a_t,'b':b_t,'label':1})
            p1.append({'goal':goal,'a':b_t,'b':a_t,'label':0})
            p2.append({'goal':goal,'a':a_t,'b':b_t,'label':1})
        for ai, bi in incompat:
            a_t, b_t = steps[ai], steps[bi]
            p1.append({'goal':goal,'a':a_t,'b':b_t,'label':0})
            p1.append({'goal':goal,'a':b_t,'b':a_t,'label':0})
            if depths.get(ai) == depths.get(bi):
                p2.append({'goal':goal,'a':a_t,'b':b_t,'label':0})
                p2.append({'goal':goal,'a':b_t,'b':a_t,'label':0})
    return pd.DataFrame(p1), pd.DataFrame(p2)

def balance(df):
    pos = df[df.label==1]; neg = df[df.label==0]
    n = min(len(pos), len(neg))
    return pd.concat([pos.sample(n,random_state=SEED),
                      neg.sample(n,random_state=SEED)],
                     ignore_index=True
                    ).sample(frac=1,random_state=SEED).reset_index(drop=True)

p1_train_raw, p2_train_raw = build_rows(train_plans, exclude=eval_goals)
p1_train = balance(p1_train_raw)
p2_train = balance(p2_train_raw)

# Issue 18: dev uses ALL examples, not balanced — AUC doesn't need balance
p1_dev_raw, p2_dev_raw = build_rows(dev_plans)
p1_dev = p1_dev_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
p2_dev = p2_dev_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Probe 1 — train: {len(p1_train)} (balanced)  dev: {len(p1_dev)} (all)')
print(f'Probe 2 — train: {len(p2_train)} (balanced)  dev: {len(p2_dev)} (all)')


---
## Feature extraction

One forward pass per example captures all candidate layers. Cached to disk
with a config hash — changing model, sweep layers, or prompt invalidates the cache.


In [ ]:
def extract(df, tag):
    if cfg['use_cache'] and has_cache(tag):
        print(f'  Loading cache: {_cpath(tag).name}')
        return load_cache(tag)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=tag):
        hiddens, p_yes = get_all_layers_and_pyes(
            tp1_ordering(r.goal, r.a, r.b))
        rows.append({'hiddens':hiddens, 'p_yes':p_yes,
                     'label':r.label, 'goal':r.goal})
    save_cache(tag, rows)
    return rows

p1_tr_feats = extract(p1_train, 'p1_train')
p2_tr_feats = extract(p2_train, 'p2_train')
p1_dv_feats = extract(p1_dev,   'p1_dev')
p2_dv_feats = extract(p2_dev,   'p2_dev')
print(f'Features: {len(p1_tr_feats)} p1_train  {len(p1_dv_feats)} p1_dev')


---
## Layer sweep

Probes are **fitted on training features** and **evaluated on dev features**.
Dev AUC is computed on the full unbalanced dev set (issue 18 fix).
Each probe selects its best layer independently.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Issue 20: StandardScaler + more iterations, no global warning suppression
def make_probe():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=5000, class_weight='balanced'))

def sweep(tr_feats, dv_feats, name):
    y_tr = np.array([r['label'] for r in tr_feats])
    y_dv = np.array([r['label'] for r in dv_feats])
    # Issue 19: also compute per-plan AUC on dev
    dv_goals = np.array([r['goal'] for r in dv_feats])
    rows, best_auc, best_layer = [], -1, None

    for li, layer in enumerate(cfg['sweep_layers']):
        X_tr = np.array([r['hiddens'][li] for r in tr_feats])
        X_dv = np.array([r['hiddens'][li] for r in dv_feats])
        probe = make_probe()
        probe.fit(X_tr, y_tr)
        proba_dv = probe.predict_proba(X_dv)[:, 1]
        dev_auc  = roc_auc_score(y_dv, proba_dv)
        # Issue 19: macro per-plan AUC
        plan_aucs = []
        for g in set(dv_goals):
            mask = dv_goals == g
            if len(set(y_dv[mask])) < 2: continue
            plan_aucs.append(roc_auc_score(y_dv[mask], proba_dv[mask]))
        macro_auc = float(np.mean(plan_aucs)) if plan_aucs else float('nan')
        marker = '  ← best' if dev_auc > best_auc else ''
        print(f'  Layer {layer:2d}: dev AUC {dev_auc:.4f}  '
              f'macro {macro_auc:.4f}{marker}')
        rows.append({'layer':layer,'dev_auc':round(dev_auc,4),
                     'macro_auc':round(macro_auc,4)})
        if dev_auc > best_auc:
            best_auc, best_layer = dev_auc, layer

    df = pd.DataFrame(rows)
    df.to_csv(CONTENT / f'{name}_layer_sweep.csv', index=False)
    return df, best_layer, best_auc

print('=== Probe 1 layer sweep ===')
p1_sweep, p1_best, p1_auc = sweep(p1_tr_feats, p1_dv_feats, 'probe1')
print(f'Best: layer {p1_best}  AUC {p1_auc:.4f}\n')

print('=== Probe 2 layer sweep ===')
p2_sweep, p2_best, p2_auc = sweep(p2_tr_feats, p2_dv_feats, 'probe2')
print(f'Best: layer {p2_best}  AUC {p2_auc:.4f}')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sw, best, title in [
    (axes[0], p1_sweep, p1_best, 'Probe 1 (Mode 1)'),
    (axes[1], p2_sweep, p2_best, 'Probe 2 (Mode 2)')]:
    ax.plot(sw.layer, sw.dev_auc, 'o-', color='#185FA5', lw=2, ms=5,
            label='micro AUC')
    ax.plot(sw.layer, sw.macro_auc, 's--', color='#1D9E75', lw=1.5, ms=4,
            label='macro per-plan AUC')
    ax.axvline(best, color='#E24B4A', lw=1.5, ls='--', label=f'Best: {best}')
    ax.set_xlabel('Layer'); ax.set_ylabel('Dev AUC')
    ax.set_title(title); ax.legend(fontsize=9); ax.set_ylim(0.5,1.0)
plt.suptitle('Layer sweep — fit:train | eval:dev', y=1.02)
plt.tight_layout()
plt.savefig(CONTENT/'layer_sweep.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
import pickle, json as _json

def train_final(tr_feats, best_layer, name):
    li = cfg['sweep_layers'].index(best_layer)
    X  = np.array([r['hiddens'][li] for r in tr_feats])
    y  = np.array([r['label']       for r in tr_feats])
    probe = make_probe()
    probe.fit(X, y)
    # Issue 20: check convergence
    lr = probe.named_steps['logisticregression']
    if lr.n_iter_[0] >= lr.max_iter:
        print(f'  WARNING: {name} did not converge in {lr.max_iter} iterations')
    # Issue 13: save dict with metadata
    out = CONTENT / f'{name}_layer{best_layer}.pkl'
    pickle.dump({
        'probe': probe,
        'layer': best_layer,
        'model_name': cfg['model_name'],
        'sweep_layers': cfg['sweep_layers'],
        'n_train': len(X),
        'feature_dim': X.shape[1],
    }, open(out, 'wb'))
    print(f'  Saved {out.name}')
    return probe

probe1 = train_final(p1_tr_feats, p1_best, 'probe1')
probe2 = train_final(p2_tr_feats, p2_best, 'probe2')

# Issue 5+13: save manifest
manifest = {
    'probe1_file': f'probe1_layer{p1_best}.pkl',
    'probe1_layer': p1_best,
    'probe2_file': f'probe2_layer{p2_best}.pkl',
    'probe2_layer': p2_best,
    'model_name': cfg['model_name'],
    'cache_sig': _cache_sig,
}
_json.dump(manifest, open(CONTENT/'probe_manifest.json','w'), indent=2)
print(f'Manifest saved: {manifest}')


---
## Permutation tests

**Scope:** group-CV permutation test at the fixed layers selected on the development
set. This tests whether hidden states carry genuine label-correlated signal, not
whether a significant layer exists in the sweep range. A full max-statistic
permutation across all candidate layers would test the latter but is substantially
more expensive.

**Exchangeability note:** labels are permuted globally across rows. Plans with
many pairs contribute more rows, and within-plan pairs are correlated. A strictly
correct null would preserve within-plan structure. The current global permutation
is a standard approximation; the test remains informative as evidence that the
probe exploits feature-label correspondence rather than fitting noise.


In [ ]:
from sklearn.model_selection import GroupKFold

gkf      = GroupKFold(n_splits=cfg['n_cv_folds'])
rng_perm = np.random.default_rng(SEED)

def permutation_test(tr_feats, best_layer, name):
    li     = cfg['sweep_layers'].index(best_layer)
    X      = np.array([r['hiddens'][li] for r in tr_feats])
    y      = np.array([r['label']       for r in tr_feats])
    groups = np.array([r['goal']        for r in tr_feats])

    def cv_auc(labels):
        fold_aucs = []
        for tr, val in gkf.split(X, labels, groups=groups):
            # Issue 17: skip folds with one class
            if len(set(labels[val])) < 2: continue
            if len(set(labels[tr]))  < 2: continue
            p = make_probe()
            p.fit(X[tr], labels[tr])
            fold_aucs.append(
                roc_auc_score(labels[val], p.predict_proba(X[val])[:,1]))
        return float(np.mean(fold_aucs)) if fold_aucs else 0.5

    real_auc = cv_auc(y)
    perm_aucs = []
    for i in range(cfg['n_perms']):
        perm_aucs.append(cv_auc(rng_perm.permutation(y)))
        if (i+1) % 25 == 0:
            print(f'  [{name}] {i+1}/{cfg["n_perms"]}  '
                  f'perm mean: {np.mean(perm_aucs):.4f}')

    # Issue 16: correct p-value for finite Monte Carlo
    p_val = (1 + np.sum(np.array(perm_aucs) >= real_auc)) / (len(perm_aucs) + 1)
    pd.DataFrame({'perm_auc': perm_aucs}).to_csv(
        CONTENT / f'permutation_aucs_{name}.csv', index=False)
    return real_auc, perm_aucs, float(p_val)

print('=== Probe 1 permutation test ===')
p1_real, p1_perms, p1_pval = permutation_test(p1_tr_feats, p1_best, 'probe1')
print(f'  Real {p1_real:.4f}  perm mean {np.mean(p1_perms):.4f}  p={p1_pval:.4f}\n')

print('=== Probe 2 permutation test ===')
p2_real, p2_perms, p2_pval = permutation_test(p2_tr_feats, p2_best, 'probe2')
print(f'  Real {p2_real:.4f}  perm mean {np.mean(p2_perms):.4f}  p={p2_pval:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, real, perms, pval, name, layer in [
    (axes[0], p1_real, p1_perms, p1_pval, 'Probe 1', p1_best),
    (axes[1], p2_real, p2_perms, p2_pval, 'Probe 2', p2_best)]:
    ax.hist(perms, bins=25, color='#888780', alpha=0.7,
            label=f'Permuted ({cfg["n_perms"]}×)')
    ax.axvline(real, color='#185FA5', lw=2.5,
               label=f'Real AUC = {real:.4f}')
    ax.axvline(np.mean(perms), color='#E24B4A', lw=1.5, ls='--',
               label=f'Perm mean = {np.mean(perms):.4f}')
    ax.set_xlabel('CV AUC'); ax.set_ylabel('Count')
    ax.set_title(f'{name} (layer {layer})')
    ax.legend(fontsize=9)
    # Issue 16: correct minimum p labelling
    min_p = 1/(cfg['n_perms']+1)
    lbl = (f'p < {min_p*2:.3f}' if pval <= min_p*1.01
           else f'p = {pval:.3f}')
    ax.text(0.98, 0.97, lbl, transform=ax.transAxes,
            ha='right', va='top', fontsize=10,
            color='#185FA5' if pval < 0.05 else '#888780')
plt.tight_layout()
plt.savefig(CONTENT/'permutation_test.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
print('=== SUMMARY ===')
print(f'Probe 1 | layer {p1_best} | dev AUC {p1_auc:.4f} | '
      f'perm p={p1_pval:.4f}')
print(f'Probe 2 | layer {p2_best} | dev AUC {p2_auc:.4f} | '
      f'perm p={p2_pval:.4f}')
